# Kata 03 — Normalización con `PostToolUse`

> Spec: [`specs/003-posttool-normalize/spec.md`](../../specs/003-posttool-normalize/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json

client, settings = bootstrap(budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Una herramienta legacy devuelve XML, payload heterogéneo, códigos arcanos. Si lo paso tal cual al modelo, gasta tokens parseándolo cada turno y aluciinará si el formato cambia. Un *normalizador post-tool* corre del lado del cliente: recibe el output bruto, lo proyecta a un JSON pequeño y canónico, y eso es lo que el modelo ve.

## 2. Por qué importa

El modelo no debe ser el parser de tu sistema. Cada token de XML legacy que cargo en su contexto es un token de presupuesto perdido y una oportunidad de interpretación errónea.

## 3. Ejemplo correcto — normalizar antes de pasar al modelo

In [ ]:
RAW_LEGACY = {
    "@id": "INV-9921",
    "s": "0xA7",       # status code arcano
    "amt": "1234.56",  # string, no número
    "_meta": "<legacy_xml>...mucho ruido...</legacy_xml>",
}

STATUS_MAP = {"0xA1": "paid", "0xA7": "timeout", "0xB3": "rejected"}

def posttool_normalize(tool_name: str, raw: dict) -> dict:
    if tool_name != "legacy_lookup":
        return raw
    return {
        "id": raw["@id"],
        "status": STATUS_MAP.get(raw["s"], "unknown"),
        "amount": float(raw["amt"]),
    }

print("crudo:", RAW_LEGACY)
print("normalizado:", posttool_normalize("legacy_lookup", RAW_LEGACY))

### 3.1 El modelo razona sobre el JSON limpio

In [ ]:
LOOKUP_TOOL = {
    "name": "legacy_lookup",
    "description": "Consulta una factura en el sistema legacy. Devuelve datos normalizados.",
    "input_schema": {
        "type": "object",
        "properties": {"invoice_id": {"type": "string"}},
        "required": ["invoice_id"],
    },
}

def call_with_post_normalize(client, *, system, user_msg):
    messages = [{"role": "user", "content": user_msg}]
    resp = client.messages.create(system=system, messages=messages, tools=[LOOKUP_TOOL])
    if resp.stop_reason != "tool_use":
        return resp, None, None
    tu = next(b for b in resp.content if b.type == "tool_use")
    raw = RAW_LEGACY  # nuestro fake legacy
    clean = posttool_normalize(tu.name, raw)
    messages.append({"role": "assistant", "content": resp.content})
    messages.append({"role": "user", "content": [{
        "type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(clean),
    }]})
    final = client.messages.create(system=system, messages=messages, tools=[LOOKUP_TOOL])
    return final, raw, clean

system = "Eres un asistente. Si el usuario pregunta por una factura, llama a legacy_lookup."
final, raw, clean = call_with_post_normalize(client, system=system, user_msg="¿En qué estado está la factura INV-9921?")
print("payload del modelo:", clean)
print("respuesta:", next((b.text for b in final.content if b.type == "text"), ""))
print("input_tokens segunda llamada:", final.usage.input_tokens)

## 4. Anti-patrón — el modelo recibe el crudo

In [ ]:
def call_without_normalize(client, *, system, user_msg):
    messages = [{"role": "user", "content": user_msg}]
    resp = client.messages.create(system=system, messages=messages, tools=[LOOKUP_TOOL])
    if resp.stop_reason != "tool_use":
        return resp, None
    tu = next(b for b in resp.content if b.type == "tool_use")
    raw = RAW_LEGACY  # se inyecta tal cual
    messages.append({"role": "assistant", "content": resp.content})
    messages.append({"role": "user", "content": [{
        "type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(raw),
    }]})
    final = client.messages.create(system=system, messages=messages, tools=[LOOKUP_TOOL])
    return final, raw

final_bad, raw = call_without_normalize(client, system=system, user_msg="¿En qué estado está la factura INV-9921?")
print("payload del modelo:", raw)
print("respuesta:", next((b.text for b in final_bad.content if b.type == "text"), ""))
print("input_tokens segunda llamada:", final_bad.usage.input_tokens)

Compara los `input_tokens` y la calidad de la respuesta entre las dos celdas. Aún con un payload pequeño la diferencia es visible; en un sistema legacy real (XML de varios KB) la diferencia es de orden de magnitud.

## 5. Argumento de certificación

- **El modelo nunca debe ver datos crudos heterogéneos.** Es un anti-patrón gastar tokens en que parsee lo que un Python `dict.get()` resuelve en microsegundos.
- **El normalizador es puro y testeable** sin LLM: misma entrada → mismo JSON.
- **Códigos arcanos → strings legibles** elimina memorización del modelo (hoy `0xA7` es `timeout`; mañana puede ser otro).
- **Conexión con otros katas**: el JSON canónico que produce este hook respeta el contrato del Kata 5 (schemas) y baja el costo del Kata 10 (caching, prefijos estables).

## 6. Auto-evaluación

**1. ¿Qué pasa si el hook recibe un código que no está en el mapa?**

Devuelvo `"unknown"` para no romper el contrato (el campo siempre existe). En producción, además, registro un evento de "código no mapeado" para que el equipo lo revise. Nunca dejo que el modelo "adivine" qué significa.

**2. ¿Cómo pruebo el hook sin levantar el modelo?**

`posttool_normalize` es una función pura sobre dicts. Le paso un fixture y compruebo el resultado con `assert`. Cero llamadas a la API. Si más adelante el formato legacy cambia, agrego entradas al fixture y al `STATUS_MAP`.

**3. ¿Qué métrica concreta demuestra que el hook redujo "carga cognitiva"?**

`response.usage.input_tokens` en la segunda llamada (la que ve el `tool_result`). Comparé el run normalizado vs el crudo en §4. Para evidencia más fuerte, podría correr un eval con N invocaciones y medir tasa de respuestas correctas.